In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Buscar pass.env en la carpeta actual
env_path=Path.cwd()/"pass.env"
loaded=load_dotenv(dotenv_path=env_path,override=True)

print(f"¿Archivo encontrado y cargado?: {loaded}")

PG_CONNECTION_STRING=os.getenv("PG_CONNECTION_STRING")

if not PG_CONNECTION_STRING:
    raise ValueError("Falta PG_CONNECTION_STRING en pass.env")

print("Variables cargadas correctamente.")


¿Archivo encontrado y cargado?: True
Variables cargadas correctamente.


In [2]:
import psycopg2

conn=psycopg2.connect(PG_CONNECTION_STRING)
cur=conn.cursor()

cur.execute("SELECT current_database(),current_user,version();")
bd,usuario,version=cur.fetchone()

print("Conectado a PostgreSQL")
print("Base de datos:",bd)
print("Usuario:",usuario)
print(version)


Conectado a PostgreSQL
Base de datos: neondb
Usuario: neondb_owner
PostgreSQL 18.6 (2078fcb) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [3]:
# squema_bd.sql es la única fuente de verdad del modelo relacional.
# Este notebook lo ejecuta completo para evitar que el SQL y el notebook
# terminen creando estructuras diferentes.

schema_path=Path.cwd()/"squema_bd.sql"

if not schema_path.exists():
    raise FileNotFoundError(f"No se encontró {schema_path.name} en {Path.cwd()}")

sql_schema=schema_path.read_text(encoding="utf-8")

try:
    cur.execute(sql_schema)
    conn.commit()
    print("Esquema creado correctamente a partir de squema_bd.sql.")
except Exception:
    conn.rollback()
    print("Ocurrió un error. Se hizo rollback y no se confirmó la creación.")
    raise


Esquema creado correctamente a partir de squema_bd.sql.


In [4]:
# Verificación rápida de las tablas creadas
cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
""")

tablas=[fila[0] for fila in cur.fetchall()]

print(f"Tablas encontradas: {len(tablas)}")
for tabla in tablas:
    print("-",tabla)

esperadas={
    "roles","usuarios","auditoria_cambios","pacientes","antecedentes",
    "reportes_previos","encuentros","observaciones","diagnosticos",
    "notas_clinicas","examenes","medicamentos","prescripciones",
    "facturas","factura_detalle"
}

faltantes=esperadas-set(tablas)

if faltantes:
    raise RuntimeError(f"Faltan tablas esperadas: {sorted(faltantes)}")

print("Las 15 tablas esperadas fueron creadas correctamente.")


Tablas encontradas: 15
- antecedentes
- auditoria_cambios
- diagnosticos
- encuentros
- examenes
- factura_detalle
- facturas
- medicamentos
- notas_clinicas
- observaciones
- pacientes
- prescripciones
- reportes_previos
- roles
- usuarios
Las 15 tablas esperadas fueron creadas correctamente.


In [5]:
cur.close()
conn.close()
print("Conexión cerrada correctamente.")


Conexión cerrada correctamente.
